## Inference under FHE for the MNIST Dataset using helayers

In this demo, we'll deal with a classification problem for the MNIST dataset [1], trying to correctly classify a batch of samples using a neural network model that will be created and trained using the Keras library (with architecture similar to reference [2]).
First, we'll build a plain neural network for the MNIST model. Then, we'll encrypt the trained network and run inference over it using FHE.

In [1]:
import os

##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)
from tensorflow.keras import backend as K

from tensorflow.keras import utils, losses
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
import h5py

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)

batch_size = 16
# batch_size = 32
# batch_size = 64
# batch_size = 128
# batch_size = 256
# batch_size = 512
# batch_size = 1024
# batch_size = 2048
# batch_size = 4096
# batch_size=8192
print("Misc. initializations")

2025-07-13 15:50:48.247417: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Misc. initializations


### Load and Preprocess the MNIST Dataset. 

In [2]:
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split

# 步骤1：生成测试集文件（适配变量名）
def generate_test_set():
    # 加载原始数据集
    data = scipy.io.loadmat('/workspace/examples/vision_transformer/data/coil20/coil20.mat')
    
    # 根据实际变量名调整
    fea = data['X']  # 原'fea'替换为实际变量名
    gnd = data['Y']  # 原'gnd'替换为实际变量名
    # 在generate_test_set中：
    print("原始数据集样本数:", fea.shape[0])  # 应为1440

    
    # 分层划分训练集和测试集
    x_train, x_test, y_train, y_test = train_test_split(
        fea, gnd, 
        test_size=0.2,
        stratify=gnd,
        random_state=42
    )
    # 划分训练测试后：
    print("划分后训练集样本数:", len(x_train))  # 应为1440 * 0.8=1152
    print("划分后测试集样本数:", len(x_test))   # 应为288
    print("打印形状:", x_train.shape)  
    print("打印形状:", x_test.shape)   
    
    # 保存时使用一致的变量名
    scipy.io.savemat('/workspace/examples/vision_transformer/data/coil20/coil20_train.mat',
                    {'X': x_train, 'Y': y_train})
    scipy.io.savemat('/workspace/examples/vision_transformer/data/coil20/coil20_test.mat',
                    {'X': x_test, 'Y': y_test})  # 键名与原始数据对齐

# 步骤2：数据加载函数同步更新
def load_coil20(path):
    data = scipy.io.loadmat(path)
    images = data['X'].reshape(-1, 32, 32)  # 变量名同步修改
    labels = data['Y'].flatten() - 1
    return images, labels

# 执行生成测试集
generate_test_set()

# 加载数据（适配新变量名）
x_train, y_train = load_coil20('/workspace/examples/vision_transformer/data/coil20/coil20.mat')
x_test, y_test = load_coil20('/workspace/examples/vision_transformer/data/coil20/coil20_test.mat')
print("划分后训练集样本数:", len(x_train))  # 应为1440 * 0.8=1152
print("划分后测试集样本数:", len(x_test))   # 应为288

# 预处理流程（保持分辨率）
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# 验证输入形状
print(f"输入形状: {x_train[0].shape}")  # 应为(32,32,1)


原始数据集样本数: 1440
划分后训练集样本数: 1152
划分后测试集样本数: 288
打印形状: (1152, 1024)
打印形状: (288, 1024)
划分后训练集样本数: 1440
划分后测试集样本数: 288
输入形状: (32, 32, 1)


In [3]:
# 划分验证集（从训练集中取200个样本）
val_size = 200
x_train = x_train[:-val_size]
x_val = x_train[-val_size:]
y_train = y_train[:-val_size]
y_val = y_train[-val_size:]

# 输出最终数据集信息
print(f"训练集形状: {x_train.shape}, 样本数: {len(y_train)}")
print(f"验证集形状: {x_val.shape}, 样本数: {len(y_val)}")
print(f"测试集形状: {x_test.shape}, 样本数: {len(y_test)}")
print(f"输入形状: {x_train[0].shape}")

训练集形状: (1240, 32, 32, 1), 样本数: 1240
验证集形状: (200, 32, 32, 1), 样本数: 200
测试集形状: (288, 32, 32, 1), 样本数: 288
输入形状: (32, 32, 1)


### Save Dataset

In [4]:
def save_data_set(x, y, data_type, s=''):
    fname=os.path.join(PATH, f'x_{data_type}{s}.h5')
    print("Saving x_{} of shape {} in {}".format(data_type, x.shape,fname))
    xf = h5py.File(fname, 'w')
    xf.create_dataset('x_{}'.format(data_type), data=x)
    xf.close()

    yf = h5py.File(os.path.join(PATH, f'y_{data_type}{s}.h5'), 'w')
    yf.create_dataset(f'y_{data_type}', data=y)
    yf.close()

save_data_set(x_test, y_test, data_type='test10')
save_data_set(x_train, y_train, data_type='train')
save_data_set(x_val, y_val, data_type='val')

Saving x_test10 of shape (288, 32, 32, 1) in data/net_mnist/x_test10.h5
Saving x_train of shape (1240, 32, 32, 1) in data/net_mnist/x_train.h5
Saving x_val of shape (200, 32, 32, 1) in data/net_mnist/x_val.h5


### Build a Plain Neural Network for the MNIST Model

In [5]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
# 创建模型
model = Sequential()

# 输入层：适应32x32分辨率
model.add(Conv2D(20, kernel_size=5, kernel_regularizer=l2(0.0005), input_shape=(32, 32, 1)))
model.add(AvgPool2D(pool_size=2))  # 输出: 14x14x20 (∵ (32-5+1)/2 = 14)

# 中间卷积层
model.add(Conv2D(50, kernel_size=5, kernel_regularizer=l2(0.0005)))
model.add(AvgPool2D(pool_size=2))  # 输出: 5x5x50 (∵ (14-5+1)/2 = 5)

# 关键修改：输出20个类别
model.add(Conv2D(20, kernel_size=(5,5), kernel_regularizer=l2(0.0005)))  # 输出: 1x1x20 (∵ 5-5+1=1)
model.add(BatchNormalization())
model.add(ReLU())

# 维度处理
model.add(AvgPool2D(pool_size=(1,1)))  # 维持1x1空间维度
model.add(Flatten())                   # 输出形状: (20,)

# 输出模型架构

model.summary()

2025-07-13 15:50:54.868905: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 14, 14, 20)       0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 10, 10, 50)        25050     
                                                                 
 average_pooling2d_1 (Averag  (None, 5, 5, 50)         0         
 ePooling2D)                                                     
                                                                 
 conv2d_2 (Conv2D)           (None, 1, 1, 20)          25020     
                                                                 
 batch_normalization (BatchN  (None, 1, 1, 20)         8

2025-07-13 15:50:54.978351: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:50:54.981013: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-13 15:50:54.985102: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-13 15:50:54.986604: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so ret

### Train the Neural Network

In [6]:
#一阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_addons as tfa

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=20,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
# def lr_scheduler(epoch):
#     if epoch < 100:
#         return 0.001
#     elif epoch < 200:
#         return 0.0005
#     else:
#         return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

model.fit(
    x_train, y_train, batch_size=batch_size,
    
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.9.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make su

Epoch 1/4000


2025-07-13 15:51:00.160032: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8500


78/78 - 6s - loss: 3.0073 - accuracy: 0.2927 - val_loss: 3.0195 - val_accuracy: 0.2000 - 6s/epoch - 73ms/step
Epoch 2/4000
78/78 - 0s - loss: 2.8150 - accuracy: 0.4500 - val_loss: 2.9503 - val_accuracy: 0.3600 - 241ms/epoch - 3ms/step
Epoch 3/4000
78/78 - 0s - loss: 2.3559 - accuracy: 0.6032 - val_loss: 2.5053 - val_accuracy: 0.3600 - 245ms/epoch - 3ms/step
Epoch 4/4000
78/78 - 0s - loss: 2.1130 - accuracy: 0.6952 - val_loss: 2.3890 - val_accuracy: 0.3600 - 251ms/epoch - 3ms/step
Epoch 5/4000
78/78 - 0s - loss: 1.9987 - accuracy: 0.7379 - val_loss: 2.2055 - val_accuracy: 0.7200 - 254ms/epoch - 3ms/step
Epoch 6/4000
78/78 - 0s - loss: 1.9294 - accuracy: 0.7573 - val_loss: 1.7755 - val_accuracy: 0.7200 - 250ms/epoch - 3ms/step
Epoch 7/4000
78/78 - 0s - loss: 1.8856 - accuracy: 0.7581 - val_loss: 1.4425 - val_accuracy: 0.7200 - 255ms/epoch - 3ms/step
Epoch 8/4000
78/78 - 0s - loss: 1.8087 - accuracy: 0.7677 - val_loss: 1.2740 - val_accuracy: 0.7200 - 247ms/epoch - 3ms/step
Epoch 9/4000
78

### Rebuild Model

In [7]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
new_model = Sequential()
for layer in model.layers:
    
    if isinstance(layer, ReLU):
        new_model.add(SquareActivation())
    else:
        new_layer = layer.__class__.from_config(layer.get_config())
        new_model.add(new_layer)
        if layer.weights:
            new_layer.set_weights(layer.get_weights())  # 自动复制权重和偏置

new_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 14, 14, 20)       0         
 ooling2D)                                                       
                                                                 
 conv2d_1 (Conv2D)           (None, 10, 10, 50)        25050     
                                                                 
 average_pooling2d_1 (Averag  (None, 5, 5, 50)         0         
 ePooling2D)                                                     
                                                                 
 conv2d_2 (Conv2D)           (None, 1, 1, 20)          25020     
                                                                 
 batch_normalization (BatchN  (None, 1, 1, 20)        

In [8]:
#二阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=20,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
# def lr_scheduler(epoch):
#     if epoch < 100:
#         return 0.001
#     elif epoch < 200:
#         return 0.0005
#     else:
#         return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

new_model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

new_model.fit(
    x_train, y_train, batch_size=batch_size,
   
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = new_model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000
78/78 - 1s - loss: 0.1839 - accuracy: 0.9911 - val_loss: 0.0792 - val_accuracy: 1.0000 - 848ms/epoch - 11ms/step
Epoch 2/4000
78/78 - 0s - loss: 0.1717 - accuracy: 0.9919 - val_loss: 0.0778 - val_accuracy: 1.0000 - 246ms/epoch - 3ms/step
Epoch 3/4000
78/78 - 0s - loss: 0.1787 - accuracy: 0.9919 - val_loss: 0.0775 - val_accuracy: 1.0000 - 244ms/epoch - 3ms/step
Epoch 4/4000
78/78 - 0s - loss: 0.1824 - accuracy: 0.9944 - val_loss: 0.0782 - val_accuracy: 1.0000 - 243ms/epoch - 3ms/step
Epoch 5/4000
78/78 - 0s - loss: 0.1808 - accuracy: 0.9895 - val_loss: 0.0773 - val_accuracy: 1.0000 - 244ms/epoch - 3ms/step
Epoch 6/4000
78/78 - 0s - loss: 0.1794 - accuracy: 0.9935 - val_loss: 0.0766 - val_accuracy: 1.0000 - 247ms/epoch - 3ms/step
Epoch 7/4000
78/78 - 0s - loss: 0.1926 - accuracy: 0.9871 - val_loss: 0.0761 - val_accuracy: 1.0000 - 243ms/epoch - 3ms/step
Epoch 8/4000
78/78 - 0s - loss: 0.1743 - accuracy: 0.9903 - val_loss: 0.0751 - val_accuracy: 1.0000 - 243ms/epoch - 3ms/step

### Serialize Model and Weights

In [9]:
model_json = new_model.to_json()
with open(os.path.join(PATH, 'model617.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
new_model.save_weights(os.path.join(PATH, 'model617.h5'))
print("Saved model to disk")

Saved model to disk


In [ ]:
# model_json = model.to_json()
# with open(os.path.join(PATH, 'model215.json'), "w") as json_file:
#     json_file.write(model_json)
# # serialize weights to HDF5
# model.save_weights(os.path.join(PATH, 'model215.h5'))
# print("Saved model to disk")

We are all done training the plain network. Next we will encrypt the network and run inference over it using FHE. Let's start with some initializations.

In [ ]:
import pyhelayers
import utils

utils.verify_memory()

print('Misc. initializations')

In [ ]:
import pyhelayers
import utils
import h5py
import os
import numpy as np
##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
# from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)



utils.verify_memory()

print('Misc. initializations')

batch_size=500

The following is a general outline of the next steps.

We encode and encrypt a neural network model, using the files that were created and saved before. An automated optimizer, that occurs during the call to encode_encrypt, will examine our network and will determine various configuration details that will allow running inference under encryption efficiently.

Next, we will demonstrate how we can encrypt data, run inference on our encrypted network, and compare the results against the expected labels.
Now let's dive in . . .

In [ ]:
he_run_req = pyhelayers.HeRunRequirements()
he_run_req.set_he_context_options([pyhelayers.DefaultContext()])
he_run_req.optimize_for_batch_size(16)

nn = pyhelayers.NeuralNet()
nn.encode_encrypt([os.path.join(PATH, "model.json"), os.path.join(PATH, "model.h5")], he_run_req)

The encode_encrypt method also initialized an HeContext object containing the keys. We obtain it now from the model so we can encrypt the input data.

In [ ]:
context = nn.get_created_he_context()

We will now load real samples of the MNIST dataset to classify. We will load the samples and the corresponding true labels from HDF5 files. We will also extract the first batch of samples and labels.

In [ ]:
with h5py.File(os.path.join(PATH, "x_test.h5")) as f:
    x_test = np.array(f["x_test"])
with h5py.File(os.path.join(PATH, "y_test.h5")) as f:
    y_test = np.array(f["y_test"])
    
# plain_samples, labels = utils.extract_batch(x_test, y_test, batch_size, 0)

plain_samples, labels = utils.extract_batch(x_test, y_test, 100, 0)
print('Batch of size',batch_size,'loaded')

To populate the input, we need to encode and then encrypt the values of the plain input under HE.

In [ ]:
model_io_encoder = pyhelayers.ModelIoEncoder(nn)
samples = pyhelayers.EncryptedData(context)
model_io_encoder.encode_encrypt(samples, [plain_samples])
print('Test data encrypted')

We now go ahead with the inference itself. We run the encrypted input through the encrypted NN to obtain encrypted results. This computation does not use the secret key and acts on completely encrypted values. Running the inference is done using the "predict" method of the NN, that receives the destination 3D structure to put the result of the computation in, and the input for the inference.

In [ ]:
utils.start_timer()

predictions = pyhelayers.EncryptedData(context)
nn.predict(predictions, samples)

duration=utils.end_timer('predict')
utils.report_duration('predict per sample',duration/batch_size)

In order to assess the results of the computation, we first need to decrypt them. This is done by an IO processor that has the secret key and is capable of decrypting the ciphertext and decoding it into plaintext version of the HE computation result.

In [ ]:
plain_predictions = model_io_encoder.decrypt_decode_output(predictions)
print('predictions',plain_predictions)

Now we compare the results against the expected labels and compute the confusion matrix and the accuracy.

In [ ]:
utils.assess_results(labels, plain_predictions)

<br>

References:

<sub><sup> 1.	LeCun, Yann and Cortes, Corinna. "MNIST handwritten digit database." (2010): </sup></sub>

<sub><sup> 2.	Gilad-Bachrach, R., Dowlin, N., Laine, K., Lauter, K., Naehrig, M. &amp; Wernsing, J.. (2016). CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy. Proceedings of The 33rd International Conference on Machine Learning, in Proceedings of Machine Learning Research 48:201-210 Available from https://proceedings.mlr.press/v48/gilad-bachrach16.html.
</sup></sub>
